In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os

In [2]:
url = "https://www.saramin.co.kr/zf_user/search?search_area=main&search_done=y&search_optional_item=n&searchType=search&searchword=데이터분석"
headers = {
    "User-Agent": "Mozilla/5.0"
}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

In [13]:
job_posts = soup.select("div.item_recruit")

company_list = []
recruit_list = []
detail_list = []
url_list = []

for post in job_posts:
    # 회사명
    company = post.select_one('div.area_corp > strong.corp_name > a')
    company_name = company.text.strip() if company else None

    # 채용공고명
    recruit_span = post.select_one('div.area_job h2.job_tit > a > span')
    recruit_title = recruit_span.text.strip() if recruit_span else None

    # 상세 정보 (job_condition 내부 span)
    info_items = post.select('div.job_condition span')
    details = [item.text.strip() for item in info_items]
    detail_str = "[" + ", ".join(details) + "]" if details else "[]"

    # 채용공고 URL
    title_tag = post.select_one('div.area_job h2.job_tit > a')
    href = title_tag['href'] if title_tag and title_tag.has_attr('href') else None
    full_url = "https://www.saramin.co.kr" + href if href else None

    # 리스트에 추가
    company_list.append(company_name)
    recruit_list.append(recruit_title)
    detail_list.append(detail_str)
    url_list.append(full_url)

In [14]:
df2 = pd.DataFrame({
    "Site": ["Saramin"] * len(company_list),
    "Col_Company": company_list,
    "Col_Recruit": recruit_list,
    "Col_detail": detail_list,
    "Col_url": url_list
})

df2.head()

,Site,Col_Company,Col_Recruit,Col_detail,Col_url
0,Saramin,넛지헬스케어(주),[캐시워크] 데이터분석 담당 채용전환형 인턴,"[서울 강남구, 신입, 대졸↑, 인턴직]",https://www.saramin.co.kr/zf_user/jobs/relay/v...
1,Saramin,(주)마크클라우드,"AI 개발(Python), 데이터분석 및 사업 기획 모집","[서울 강남구, 신입·경력, 대졸↑, 정규직]",https://www.saramin.co.kr/zf_user/jobs/relay/v...
2,Saramin,(주)마크클라우드,"AI 개발(Python), 데이터분석 및 사업 기획 인턴 모집","[서울 강남구, 경력무관, 대졸↑, 인턴직]",https://www.saramin.co.kr/zf_user/jobs/relay/v...
3,Saramin,(주)씨아이템프러리,현대카드 본사 카드금융마케팅팀 사무직 채용/데이터분석/마케팅,"[서울 영등포구, 경력무관, 학력무관, 계약직]",https://www.saramin.co.kr/zf_user/jobs/relay/v...
4,Saramin,한국평가데이터(주),[서비스운영 및 데이터분석] 2025년 한국평가데이터 계약직원,"[서울 영등포구, 경력무관, 대졸↑, 계약직]",https://www.saramin.co.kr/zf_user/jobs/relay/v...


In [15]:
os.makedirs("data_tmp", exist_ok=True)
df2.to_csv("data_tmp/data_saramin.csv", index=False, encoding="utf-8-sig")